In [1]:
import numpy as np
import pandas as pd

from cl_utils import *

In [2]:
cl = pd.read_json("data/scotus_dockets.json")
cl.head()

,docket_id,docket_number,cluster_id,case_name,date_filed
0,7470,290,105849,Securities & Exchange Commission v. Variable A...,1959-03-23
1,1269,79-168,110165,"Strycker's Bay Neighborhood Council, Inc. v. K...",1980-01-07
2,8948,1047,97897,Detroit United Railway v. City of Detroit,1913-05-26
3,9534,13-5380,2642828,Woodward v. Alabama,2013-11-18
4,5084,01-8966,119762,Truesdale v. United States,2002-04-15


In [3]:
len(cl)

499345

In [4]:
len(cl[cl["docket_number"].isnull()])

1042

In [5]:
len(cl[cl["docket_number"] == ""])

8501

In [6]:
cl = cl[~cl["docket_number"].isnull()]
cl = cl[cl["docket_number"] != ""]
len(cl)

489802

In [7]:
cl["cleaned_docket_numbers"] = cl["docket_number"].apply(clean_docket_numbers)

In [8]:
cl_exploded = cl.explode("cleaned_docket_numbers", ignore_index=True)
cl_exploded.head()

,docket_id,docket_number,cluster_id,case_name,date_filed,cleaned_docket_numbers
0,7470,290,105849,Securities & Exchange Commission v. Variable A...,1959-03-23,290
1,1269,79-168,110165,"Strycker's Bay Neighborhood Council, Inc. v. K...",1980-01-07,79-168
2,8948,1047,97897,Detroit United Railway v. City of Detroit,1913-05-26,1047
3,9534,13-5380,2642828,Woodward v. Alabama,2013-11-18,13-5380
4,5084,01-8966,119762,Truesdale v. United States,2002-04-15,01-8966


In [9]:
len(cl_exploded)

543518

In [10]:
cl_exploded["docket_type"] = cl_exploded["cleaned_docket_numbers"].apply(get_docket_type)
cl_exploded["docket_type"].value_counts()

docket_type
N        521040
D          7902
A          5945
Misc       3020
M          2370
OTHER      2319
O           922
Name: count, dtype: int64

In [12]:
cl_exploded[cl_exploded["docket_type"] == "OTHER"]

,docket_id,docket_number,cluster_id,case_name,date_filed,cleaned_docket_numbers,docket_type
381,8719,"341, and Nos. 342-364",2620995,"Curtis, Collins & Holbrook Co. v. United States",1923-05-21,342-364,OTHER
385,3317,Nos. 3—5,103086,Schriber-Schroth Co. v. Cleveland Trust Co.,1938-11-07,3—5,OTHER
633,2162,"NO. 239, MISC",2620730,Hirota v. MacArthur,1949-06-27,"239, MISC",OTHER
821,1892,"226 of October term, 1905",2620726,Gila Bend Reservoir & Irrigation Co. v. Gila W...,1907-04-08,226 of October term,OTHER
838,12929,BOARD OF EDUCATION OF UNION FREE SCHOOL DISTRI...,107762,Puentes v. Board of Ed. of Union Free School D...,1968-06-17,BOARD OF EDUCATION OF UNION FREE SCHOOL DISTRI...,OTHER
...,...,...,...,...,...,...,...
542111,66727839,No. 19-7903; R46-016,9366453,Brown v. United States,2020-04-02,R46-016,OTHER
542159,66727840,No. 19-7914.; R46-017,9366454,Moss v. United States,2020-04-02,R46-017,OTHER
542319,66727838,No. 18-1218.; R46-015 / OT 2019,9366452,Buchwald Capital Advisors LLC v. Sault Ste. Ma...,2020-04-02,R46-015 / OT 2019,OTHER
542574,66728158,No. 19-7626.; R46-019,9366772,Rarden v. Ohio,2020-04-21,R46-019,OTHER


In [15]:
cl_exploded[cl_exploded["docket_number"] == "No. 15–1131."]["cleaned_docket_numbers"].to_list()

['15–1131']

In [28]:
cl_exploded[["docket_number", "cleaned_docket_numbers"]][-700:-650]

,docket_number,cleaned_docket_numbers
539962,No. 19-1185.,19-1185
539963,No. 19-7973.,19-7973
539964,No. 19-8017.,19-8017
539965,23-250,23-250
539966,23-250,23-250
539967,No. 19-8232.,19-8232
539968,No. 19-8074.,19-8074
539969,24A408,24A408
539970,No. 19-7963.,19-7963
539971,23A521,23A521


In [ ]:
# Format 1: No. 19-7979. or No. 19M138. --> ['19-7979'] or ['19M138']
# Format 2: No. 09-498; No. 09-499 --> ['09-498', '09-499']
# Format 3: No. 18-1401.; R46-020 --> ['18-1401', 'R46-020']
# Format 4: No. 96-9160 (A-862) --> ['96-9160', 'A-862']
# Format 5: 21-984 or 24A78 or 145, Orig.--> keep the same
